In [1]:
pip install pandas_ta

In [2]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas_ta as ta

In [3]:
pip install torch_geometric

In [4]:
# Install and import PyG modules separately; user must have torch-geometric installed.
try:
    from torch_geometric.nn import GCNConv
except Exception as e:
    raise ImportError("Please install torch-geometric following the official instructions. Error: " + str(e))

# Transformers FinBERT
from transformers import pipeline

# yfinance for price data
import yfinance as yf

In [5]:
import yfinance as yf

In [7]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


In [8]:
# -------------------------
# Utilities
# -------------------------
def returns_from_prices(prices: np.ndarray) -> np.ndarray:
    """Compute log returns. prices shape (T, N) -> returns (T-1, N)"""
    return np.log(prices[1:] / prices[:-1] + 1e-12)

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline


from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

def finbert_sentiment_pipeline(device=-1):
    model_path = "/content/gdrive/My Drive/finbert"

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    return pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        device=device
    )

def classify_news_sentiment(news_df: pd.DataFrame, pipe) -> pd.DataFrame:
    """
    Run FinBERT sentiment on each news text and return the news_df with 'label' and 'score' columns.
    label usually one of: 'positive', 'neutral', 'negative' (model dependent); we will normalize later.
    """
    texts = news_df["title"].tolist()
    # pipe can process batches; to be safe, process in small batches
    batch_size = 16
    labels = []
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        outs = pipe(batch)
        # outs: list of {'label': 'positive', 'score': 0.98} etc.
        for o in outs:
            labels.append(o["label"])
            scores.append(float(o.get("score", 0.0)))
    df = news_df.copy().reset_index(drop=True)
    df["label"] = labels
    df["score"] = scores
    return df


def aggregate_sentiment_per_company_day(news_df_with_labels: pd.DataFrame, tickers: List[str], date_index: pd.DatetimeIndex):
    """
    For each (date, ticker) compute an aggregated sentiment score.
    Strategy:
      - Map labels to numeric: positive -> +1, neutral -> 0, negative -> -1
      - Use label * score as weighted value, average across news mentioning the ticker (normalized format) on that date.
    Returns:
      sentiment_df: DataFrame indexed by date_index (dates from price_df) with columns tickers (shape T x N)
    """

    def label_to_num(lab):
        if pd.isna(lab):
            return 0.0
        s = str(lab).lower()
        if "pos" in s:
            return 1.0
        elif "neg" in s:
            return -1.0
        else:
            return 0.0

    tmp = news_df_with_labels.copy()
    #tmp["date"] = pd.to_datetime(tmp["date"]).dt.normalize()

    # Ensure mentions are parsed lists
    if isinstance(tmp["mentions"].iloc[0], str):
        tmp["mentions"] = tmp["mentions"].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # Normalize tickers in 'mentions' — e.g. "AAPL.US" -> "AAPL"
    def normalize_ticker(name):
        if not isinstance(name, str):
            return name
        return name.split(".")[0].upper().strip()

    tmp["mentions"] = tmp["mentions"].apply(lambda lst: [normalize_ticker(x) for x in lst])

    # Aggregate sentiment values
    accum = {}
    for _, row in tmp.iterrows():
        d = row["date"]
        numeric = label_to_num(row["label"]) * float(row["score"])
        for t in row["mentions"]:
            if t in tickers:  # only keep known companies
                accum.setdefault((d, t), []).append(numeric)

    # Build sentiment matrix
    rows = []
    for d in date_index:
        row_vals = []
        for t in tickers:
            vals = accum.get((d, t), [])
            row_vals.append(float(np.mean(vals)) if vals else 0.0)
        rows.append(row_vals)

    sentiment_df = pd.DataFrame(rows, index=date_index, columns=tickers).astype(np.float32)
    return sentiment_df


# -------------------------
# Graph construction from news co-occurrence
# -------------------------

def normalize_ticker(name):
    if not isinstance(name, str):
        return name
    return name.split(".")[0].upper().strip()




In [9]:
def return_correlation_graph_for_day(
    returns_df: pd.DataFrame,
    tickers: List[str],
    target_idx: int,
    window: int = 5,
    min_edge_weight: float = 0.001,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build an edge index and weight matrix based on the absolute Pearson correlation
    of stock returns over a rolling window ending at target_idx.
    """
    n = len(tickers)

    # 1. Slice the rolling window up to the target index
    start_idx = max(0, target_idx - window + 1)
    window_returns = returns_df.iloc[start_idx : target_idx + 1][tickers]

    # 2. Compute the Pearson correlation matrix
    if len(window_returns) > 2:
        # fillna(0.0) handles cases with zero variance (e.g., flat lined prices)
        corr_mat = window_returns.corr().fillna(0.0).values
    else:
        # Fallback if there aren't enough lookback steps yet
        corr_mat = np.eye(n, dtype=np.float32)

    # 3. Use absolute correlation to define edge strength (0 to 1)
    mat = np.abs(corr_mat)

    # 4. Filter out weak connections based on threshold
    mask = mat >= min_edge_weight
    if not np.any(mask):
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    rows, cols = np.nonzero(mask)
    edges = np.stack([rows, cols], axis=0).astype(np.int64)
    weights = mat[rows, cols].astype(np.float32)

    return edges, weights

In [14]:
class DynamicGraphDatasetNews(torch.utils.data.Dataset):
    def __init__(
        self,
        price_df: pd.DataFrame,
        news_df: pd.DataFrame,
        sentiment_df: pd.DataFrame,
        seq_len: int = 5,
        co_window: int = 5,
        mode: str = "regression",
    ):
        assert mode in ("regression", "classification")

        self.seq_len = seq_len
        self.co_window = co_window
        self.mode = mode

        prices = price_df.values.astype(np.float32)
        dates = pd.to_datetime(price_df.index)
        self.date_index = dates
        self.tickers = list(price_df.columns)

        T, N = prices.shape

        sentiment_df = sentiment_df.reindex(dates).fillna(0.0)
        svals = sentiment_df.values.astype(np.float32)

        # --------------------------------------------------
        # PRE-COMPUTE DAILY LOG RETURNS
        # --------------------------------------------------
        self.returns_df = np.log(
            (price_df + 1e-8) / (price_df.shift(1) + 1e-8)
        ).fillna(0.0)

        returns = self.returns_df.values.astype(np.float32)

           # ---- PRICE NORMALIZATION  ----
        W = 252
        VOL_WINDOW = 20
        T, N = prices.shape


        # Initialize arrays to the full length T
        norm_prices = np.zeros((T, N), dtype=np.float32)
        vol_norm = np.zeros((T, N), dtype=np.float32)
        targets = np.zeros((T, N), dtype=np.float32)

        # ---- PROCESSING ALL TIMESTEPS ----
        for t in range(T):
            # 1. Determine the window bounds
            # If t < W, use everything from 0 to t (Expanding)
            # If t >= W, use t-W+1 to t (Rolling)
            start_idx = max(0, t - W + 1)
            window = prices[start_idx : t + 1]

            # 2. Calculate statistics
            mean_t = window.mean(axis=0)
            std_t = window.std(axis=0) + 1e-6

            # 3. Normalize Current Price
            norm_prices[t] = (prices[t] - mean_t) / std_t

            # 4. Normalize Target Price (Price at t+1)
            # We can only do this if t < T-1
            if t < T - 1:
                targets[t] = (prices[t+1] - mean_t) / std_t


          # ----- Volatility -----
            vol_start = max(0, t - VOL_WINDOW + 1)
            return_window = returns[vol_start:t+1]

            # Annualize
            current_vol = return_window.std(axis=0) * np.sqrt(252) # Shape (N,)

            vol_norm[t] = current_vol

        # --------------------------------------------------
        # FEATURE CONSTRUCTION
        # --------------------------------------------------
        feature_list = []

        for t in range(T - 1):

            feat_t = np.stack(
                [
                    norm_prices[t],   # Normalized price
                    svals[t],         # Sentiment
                    vol_norm[t],      # Rolling volatility
                ],
                axis=1,
            )

            feature_list.append(feat_t.astype(np.float32))

        self.features = np.stack(feature_list, axis=0)
        self.targets = targets[:T - 1]

        self.valid_end_idx = list(range(self.seq_len - 1, T - 1))

        # --------------------------------------------------
        # GRAPH CONSTRUCTION (RETURN CORRELATION)
        # --------------------------------------------------
        self.edge_index_list = []
        self.edge_weight_list = []

        for t in range(T - 1):

            ei, ew = return_correlation_graph_for_day(
                returns_df=self.returns_df,
                tickers=self.tickers,
                target_idx=t,
                window=self.co_window,
                min_edge_weight=0.05,
            )

            # ------------------------------------------
            # EDGE WEIGHT NORMALIZATION
            # ------------------------------------------
            if ew is not None and len(ew) > 0:

                ew = ew.astype(np.float32)

                # Correlation is already bounded in [0,1]
                # Only perform Min-Max normalization
                #ew_min = ew.min()
                #ew_max = ew.max()

                #if ew_max > ew_min:
                    #ew = (ew - ew_min) / (ew_max - ew_min)
                #else:
                    #ew = np.zeros_like(ew)

                # Avoid zero-weight edges
                #ew = 0.1 + 0.9 * ew

            self.edge_index_list.append(ei)
            self.edge_weight_list.append(ew)

    def __len__(self):
        return len(self.valid_end_idx)

    def __getitem__(self, idx):

        end_t = self.valid_end_idx[idx]
        start_t = end_t - (self.seq_len - 1)

        seq_feats = self.features[start_t:end_t + 1]
        seq_edge_idx = self.edge_index_list[start_t:end_t + 1]
        seq_edge_w = self.edge_weight_list[start_t:end_t + 1]

        target = self.targets[end_t]

        if self.mode == "classification":
            y = (target > 0).astype(np.int64)
        else:
            y = target.astype(np.float32)

        return {
            "seq_feats": torch.from_numpy(seq_feats),
            "seq_edge_index": seq_edge_idx,
            "seq_edge_weight": seq_edge_w,
            "target": torch.from_numpy(y),
        }

In [15]:


def collate_dynamic(batch):
    seq_feats = torch.stack([item["seq_feats"] for item in batch], dim=0)
    targets = torch.stack([item["target"] for item in batch], dim=0)
    seq_edge_index = [item["seq_edge_index"] for item in batch]
    seq_edge_weight = [item["seq_edge_weight"] for item in batch]

    return {
        "seq_feats": seq_feats,
        "seq_edge_index": seq_edge_index,
        "seq_edge_weight": seq_edge_weight,
        "target": targets,
    }

class TGCN(nn.Module):
    def __init__(self, in_feats, gcn_hidden=64, gru_hidden=64, out_dim=1, dropout=0.2, mode="regression"):
        super().__init__()
        self.mode = mode

        self.gcn1 = GCNConv(in_feats, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(gcn_hidden, gru_hidden, batch_first=False)

        self.mlp = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Linear(gru_hidden // 2, out_dim),
        )

    def forward(self, seq_feats, seq_edge_index, seq_edge_weight):
        batch, seq_len, N, F_dim = seq_feats.shape
        device = seq_feats.device

        gcn_outputs = []

        for t in range(seq_len):
            batch_node_embeds = []

            for b in range(batch):
                x = seq_feats[b, t].to(device)

                ei_np = seq_edge_index[b][t]
                ew_np = seq_edge_weight[b][t]

                if ei_np is None or len(ei_np) == 0:
                    edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
                    edge_weight = None
                else:
                    edge_index = torch.from_numpy(ei_np).long().to(device)

                    if ew_np is not None:
                        edge_weight = torch.from_numpy(ew_np).float().to(device)
                    else:
                        edge_weight = None

                h = torch.relu(self.gcn1(x, edge_index, edge_weight))
                h = self.dropout(h)
                h = torch.relu(self.gcn2(h, edge_index, edge_weight))

                batch_node_embeds.append(h)

            gcn_outputs.append(torch.stack(batch_node_embeds, dim=0))

        seq_stack = torch.stack(gcn_outputs, dim=0)  # (seq_len, batch, N, hidden)
        seq_flat = seq_stack.view(seq_len, batch * N, -1)

        gru_out, _ = self.gru(seq_flat)
        last = gru_out[-1]

        preds = self.mlp(last)
        preds = preds.view(batch, N, -1)

        if self.mode == "classification":
            preds = torch.sigmoid(preds)

        return preds.squeeze(-1)

def train_epoch(model, loader, optimizer, device, loss_fn):
    model.train()
    total_loss = 0.0
    for batch in loader:
        seq_feats = batch["seq_feats"].to(device)
        seq_edge_index = batch["seq_edge_index"]
        seq_edge_weight = batch["seq_edge_weight"]
        target = batch["target"].to(device)
        optimizer.zero_grad()
        preds = model(seq_feats, seq_edge_index, seq_edge_weight)
        loss = loss_fn(preds, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * seq_feats.size(0)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, device, loss_fn, mode="regression"):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            seq_feats = batch["seq_feats"].to(device)
            seq_edge_index = batch["seq_edge_index"]
            seq_edge_weight = batch["seq_edge_weight"]
            target = batch["target"].to(device)
            preds = model(seq_feats, seq_edge_index, seq_edge_weight)
            loss = loss_fn(preds, target)
            total_loss += loss.item() * seq_feats.size(0)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    preds = np.concatenate(all_preds, axis=0)
    targ = np.concatenate(all_targets, axis=0)

    if mode == "classification":
        bin_preds = (preds > 0.5).astype(int)
        acc = (bin_preds == targ).mean()
        return avg_loss, {"accuracy": acc}, preds, targ
    else:
        mse = np.mean((preds - targ) ** 2)
        mae = np.mean(np.abs(preds - targ))
        return avg_loss, {"mse": mse, "mae": mae}, preds, targ


def full_news_demo(
    tickers: List[str] = None,
    start_date: str = apple_data["date"].min(),
    end_date: str = apple_data["date"].max(),
    n_news: int = len(apple_data),
    seq_len: int = 5,
    co_window: int = 5,
    batch_size: int = 16,
    epochs: int = 6,
    device_str: str = "cpu",
):
    device = torch.device(device_str)
    if tickers is None:
        # default small set (pick companies with yfinance tickers)
        tickers = ["AAPL", "AMZN", "GOOGL", "TSLA", "NFLX", "MSFT", "META", "005930.KS", "CMCSA", "IT"]

    print("Downloading price data with yfinance...")
    price_df = yf.download(tickers, start=start_date, end=end_date, progress=False,auto_adjust=False)["Close"]
    # yfinance returns multi-column if multiple tickers; ensure DataFrame shape (T, N)
    if isinstance(price_df.columns, pd.MultiIndex):
        # some tickers may be missing; flatten or select 'Adj Close' level
        price_df = price_df.copy()
    #price_df = price_df.dropna(how="all").ffill().dropna(axis=1)  # drop tickers with no data
    price_df = price_df.fillna(0)
    # ensure tickers order aligns
    price_df = price_df[tickers]
    print(f"Price data shape: {price_df.shape}")

    # Generate synthetic news
    print("Generating synthetic news...")
    news_df = apple_data[['date','title', 'symbols']] # Include symbols for mentions
    news_df.rename(columns={'symbols': 'mentions'}, inplace=True)
    # run FinBERT sentiment classification (this will download model the first time)
    print("Loading FinBERT pipeline (this may take a while for first run)...")
    pipe = finbert_sentiment_pipeline()  # cpu; set device=0 for GPU
    print("Classifying synthetic news with FinBERT...")
    news_labeled = classify_news_sentiment(news_df, pipe)
    news_labeled.reset_index(inplace=True, drop=True) # Reset index to make date a column again
    print("Sample labeled news:")
    print(news_labeled.head())

    # Aggregate sentiment per date-company
    print("Aggregating sentiment per company-day...")
    # create date_index for price_df
    date_index = price_df.index.date
    sentiment_df = aggregate_sentiment_per_company_day(news_labeled, list(price_df.columns), date_index)
    print("Sentiment df head:")
    print(sentiment_df.head())

    # build dataset
    print("Building DynamicGraphDatasetNews...")
    dataset = DynamicGraphDatasetNews(price_df=price_df, news_df=news_labeled, sentiment_df=sentiment_df, seq_len=seq_len, co_window=co_window, mode="regression")
    n = len(dataset)
    train_n = int(0.7 * n)
    val_n = int(0.15 * n)
    idxs = list(range(n))
    train_idx = idxs[:train_n]
    val_idx = idxs[train_n : train_n + val_n]
    test_idx = idxs[train_n + val_n :]

    from torch.utils.data import Subset, DataLoader

    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True, collate_fn=collate_dynamic)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)

    in_feats = dataset.features.shape[-1]
    print(f"in_feats={in_feats}, dataset length={len(dataset)}")
    model = TGCN(in_feats=in_feats, gcn_hidden=64, gru_hidden=64, out_dim=1, dropout=0.2, mode="regression").to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, device, loss_fn)
        val_loss, val_metrics, val_pred, val_targ = eval_epoch(model, val_loader, device, loss_fn, mode="regression")
        print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val Metrics: {val_metrics}")

    test_loss, test_metrics, preds, targ = eval_epoch(model, test_loader, device, loss_fn, mode="regression")
    print(f"Test Loss: {test_loss:.6f} | Test Metrics: {test_metrics}")

    return model, dataset, price_df, news_labeled, sentiment_df, preds, targ